In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/iacofi_clean.csv")
print(f"Loaded: {df.shape}")

Loaded: (4862, 222)


In [8]:
# ── FIXED SCORE CONSTRUCTION ─────────────────────────────
# Key fix: use NaN-aware scoring so missing answers don't count as correct

def score(condition):
    """Returns 1.0 if condition true, 0.0 if false, NaN if input was NaN"""
    return condition.astype(float)

# ── KNOWLEDGE (0-7) ──────────────────────────────────────
k = pd.DataFrame({
    'k_inflation':   (df['qk3'] == 3),
    'k_interest':    (df['qk4'] == 0),
    'k_simple':      (df['qk5'] == 102),
    'k_compound':    (df['qk6'] == 1),
    'k_risk_return': (df['qk7_1'] == 1),
    'k_inflation2':  (df['qk7_2'] == 1),
    'k_divers':      (df['qk7_3'] == 1),
}).astype(float)

# Replace: where original was NaN, score should be NaN not 0
for col, src in zip(k.columns, ['qk3','qk4','qk5','qk6','qk7_1','qk7_2','qk7_3']):
    k[col] = k[col].where(df[src].notna(), np.nan)

df['knowledge_score'] = k.mean(axis=1) * 7  # scale to 0-7

# ── BEHAVIOUR (0-9) ──────────────────────────────────────
b_raw = pd.DataFrame({
    'b_budget':   (df['qf2_1'] == 1),
    'b_save':     (df['qf3_98'] == 0),
    'b_shock':    (df['qf4'].isin([1,2])),
    'b_bills':    (df['qs2_5'] <= 2),
    'b_consider': (df['qs2_3'] <= 2),
    'b_leftover': (df['qs2_4'] <= 2),
    'b_goals':    (df['qs1_8'] <= 2),
    'b_watch':    (df['qs1_5'] <= 2),
    'b_notspend': (df['qs1_1'] >= 4),
}).astype(float)

for col, src in zip(b_raw.columns,
    ['qf2_1','qf3_98','qf4','qs2_5','qs2_3','qs2_4','qs1_8','qs1_5','qs1_1']):
    b_raw[col] = b_raw[col].where(df[src].notna(), np.nan)

df['behaviour_score'] = b_raw.sum(axis=1, min_count=5)  # need at least 5 valid answers

# ── ATTITUDES (0-5) ──────────────────────────────────────
a_raw = pd.DataFrame({
    'a_future':     (df['qs1_3'] >= 4),
    'a_satisfied':  (df['qs1_4'] <= 2),
    'a_goals':      (df['qs1_8'] <= 2),
    'a_noworry':    (df['qs2_1'] >= 4),
    'a_notcontrol': (df['qs2_2'] >= 4),
}).astype(float)

for col, src in zip(a_raw.columns, ['qs1_3','qs1_4','qs1_8','qs2_1','qs2_2']):
    a_raw[col] = a_raw[col].where(df[src].notna(), np.nan)

df['attitudes_score'] = a_raw.sum(axis=1, min_count=3)





In [9]:
# ── COMPOSITE ────────────────────────────────────────────
df['literacy_score'] = df['knowledge_score'] + df['behaviour_score'] + df['attitudes_score']

In [10]:
# ── SUMMARY ──────────────────────────────────────────────
print("="*45)
print(f"  Knowledge  (0-7):  {df['knowledge_score'].mean():.2f} ± {df['knowledge_score'].std():.2f}")
print(f"  Behaviour  (0-9):  {df['behaviour_score'].mean():.2f} ± {df['behaviour_score'].std():.2f}")
print(f"  Attitudes  (0-5):  {df['attitudes_score'].mean():.2f} ± {df['attitudes_score'].std():.2f}")
print(f"  TOTAL      (0-21): {df['literacy_score'].mean():.2f} ± {df['literacy_score'].std():.2f}")
print("="*45)

print("\n📍 Mean literacy score by region:")
print(df.groupby('region_name')['literacy_score']
        .mean().round(2).sort_values())

print("\n📍 Check score ranges look sensible:")
print(df[['knowledge_score','behaviour_score',
          'attitudes_score','literacy_score']].describe().round(2))

  Knowledge  (0-7):  5.02 ± 1.56
  Behaviour  (0-9):  4.81 ± 1.98
  Attitudes  (0-5):  1.77 ± 1.31
  TOTAL      (0-21): 11.70 ± 3.18

📍 Mean literacy score by region:
region_name
South         11.38
Islands       11.64
North-East    11.75
Center        11.78
North-West    11.91
Name: literacy_score, dtype: float64

📍 Check score ranges look sensible:
       knowledge_score  behaviour_score  attitudes_score  literacy_score
count          4613.00          4632.00          4528.00         4373.00
mean              5.02             4.81             1.77           11.70
std               1.56             1.98             1.31            3.18
min               0.00             0.00             0.00            0.00
25%               4.00             3.00             1.00            9.83
50%               5.00             5.00             2.00           12.00
75%               6.00             6.00             3.00           14.00
max               7.00             9.00             5.00       

In [11]:
# ── Find WHERE the real gaps actually are ─────────────────

print("="*50)
print("LITERACY SCORE BY KEY DEMOGRAPHICS")
print("="*50)

# 1. Gender
gender_map = {1: 'Male', 0: 'Female'}
print("\n📊 By Gender:")
print(df.groupby(df['qd1'].map(gender_map))['literacy_score'].mean().round(2))

# 2. Education (grouped)
edu_map = {
    1:'Postgrad', 2:'Masters', 3:'Bachelor', 4:'Some uni',
    5:'High school', 6:'Some HS', 7:'Middle school',
    8:'Some MS', 9:'Primary', 10:'No education'
}
edu_grouped = {
    1:'University+', 2:'University+', 3:'University+', 4:'University+',
    5:'High school', 6:'High school',
    7:'Middle school or below', 8:'Middle school or below',
    9:'Middle school or below', 10:'Middle school or below'
}
print("\n📊 By Education level:")
print(df.groupby(df['qd9'].map(edu_grouped))['literacy_score'].mean().round(2).sort_values())

# 3. Income
income_map = {1:'Low (≤1750€)', 2:'Mid (1751-2900€)', 3:'High (>2900€)'}
print("\n📊 By Income band:")
print(df.groupby(df['qd13'].map(income_map))['literacy_score'].mean().round(2).sort_values())

# 4. Internet access
internet_map = {1:'Has internet', 0:'No internet'}
print("\n📊 By Internet access:")
print(df.groupby(df['qd14'].map(internet_map))['literacy_score'].mean().round(2))

# 5. Age groups
df['age_group'] = pd.cut(df['qd7'],
    bins=[17,29,39,49,59,69,79],
    labels=['18-29','30-39','40-49','50-59','60-69','70-79'])
print("\n📊 By Age group:")
print(df.groupby('age_group')['literacy_score'].mean().round(2).sort_values())

# 6. Work status
work_map = {
    1:'Self-employed', 2:'Employed', 3:'Apprentice',
    4:'Homemaker', 5:'Unemployed', 6:'Retired',
    7:'Unable to work', 8:'Not working', 9:'Student', 10:'Other'
}
print("\n📊 By Work status:")
print(df.groupby(df['qd10'].map(work_map))['literacy_score'].mean().round(2).sort_values())

LITERACY SCORE BY KEY DEMOGRAPHICS

📊 By Gender:
qd1
Female    11.68
Male      11.72
Name: literacy_score, dtype: float64

📊 By Education level:
qd9
Middle school or below    11.35
High school               11.52
University+               12.26
Name: literacy_score, dtype: float64

📊 By Income band:
qd13
Low (≤1750€)        11.39
Mid (1751-2900€)    11.97
High (>2900€)       13.01
Name: literacy_score, dtype: float64

📊 By Internet access:
qd14
Has internet    11.76
No internet     10.99
Name: literacy_score, dtype: float64

📊 By Age group:
age_group
18-29    10.25
70-79    11.66
50-59    11.87
30-39    11.93
60-69    11.94
40-49    12.01
Name: literacy_score, dtype: float64

📊 By Work status:
qd10
Unemployed         9.04
Student            9.59
Not working       10.23
Homemaker         10.62
Unable to work    11.42
Employed          11.91
Apprentice        11.97
Retired           12.00
Self-employed     12.29
Name: literacy_score, dtype: float64


/tmp/ipykernel_75/2722677706.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['age_group'] = pd.cut(df['qd7'],


In [12]:
# Save
df.to_csv("../data/processed/iacofi_scored.csv", index=False)
print("\n✅ Saved → data/processed/iacofi_scored.csv")


✅ Saved → data/processed/iacofi_scored.csv


In [13]:
# Check internal consistency of the composite score
dims = df[['knowledge_score','behaviour_score','attitudes_score']].dropna()
print("Correlation between score dimensions:")
print(dims.corr().round(3))
print()
print("Interpretation:")
print("Positive moderate correlations (0.2-0.5) = dimensions are")
print("related but distinct → composite score is valid ✅")

Correlation between score dimensions:
                 knowledge_score  behaviour_score  attitudes_score
knowledge_score            1.000            0.150           -0.015
behaviour_score            0.150            1.000            0.281
attitudes_score           -0.015            0.281            1.000

Interpretation:
Positive moderate correlations (0.2-0.5) = dimensions are
related but distinct → composite score is valid ✅


In [14]:
# Compare weighted vs unweighted national mean
weighted_mean = np.average(
    df['literacy_score'].dropna(),
    weights=df.loc[df['literacy_score'].notna(), 'wght'])

print(f"Unweighted mean: {df['literacy_score'].mean():.3f}")
print(f"Weighted mean:   {weighted_mean:.3f}")
diff = abs(df['literacy_score'].mean() - weighted_mean)
print(f"Difference:      {diff:.3f} pts")
print()
print("Small difference = sampling design has minimal impact ✅")
print("Analyses on unweighted data are reliable.")

Unweighted mean: 11.703
Weighted mean:   11.519
Difference:      0.184 pts

Small difference = sampling design has minimal impact ✅
Analyses on unweighted data are reliable.
